# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [2]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [4]:
required_columns = [
    "content_hash_id",
    "client_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]

df = pd.read_parquet(
    parquet_path,
    columns=required_columns
)

print("Shape:", df.shape)







Shape: (9841378, 8)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [13]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

print("Model:", model.__class__.__name__)

Model: XGBRegressor


In [23]:
# CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Engagement rate
df["engagement_rate"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_engaged_sessions"] / df["ga4_sessions"],
    0
)

# Average engagement time
df["avg_engagement_sec"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_total_engagement_sec"] / df["ga4_sessions"],
    0
)

# Position bucket
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=False
)

df["log_impressions"] = np.log1p(df["gsc_impressions"])
print("Engineered features created.")

Engineered features created.


In [6]:
required_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

missing = [
    col for col in required_features
    if col not in df.columns
]

print("Missing features:", missing)



Missing features: []


In [21]:
print(
    df["gsc_impressions"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

count    9.841378e+06
mean     2.851812e+01
std      1.559266e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      6.000000e+00
90%      5.400000e+01
95%      1.350000e+02
99%      5.090000e+02
max      4.008400e+04
Name: gsc_impressions, dtype: float64


In [22]:

print(
    "Rows with <10 impressions:",
    (df["gsc_impressions"] < 10).sum()
)

print(
    "Rows with >=10 impressions:",
    (df["gsc_impressions"] >= 10).sum()
)

Rows with <10 impressions: 7693849
Rows with >=10 impressions: 2147529


In [24]:
df = df[df["gsc_impressions"] >= 10].reset_index(drop=True)

print("Shape after filtering low-impression rows:", df.shape)

Shape after filtering low-impression rows: (2147529, 13)


In [25]:
selected_features = [
    "gsc_impressions",
    "log_impressions",
    "gsc_avg_position",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

print("Selected features:")
print(selected_features)

Selected features:
['gsc_impressions', 'log_impressions', 'gsc_avg_position', 'engagement_rate', 'avg_engagement_sec', 'position_bucket']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df["client_hash_id"]
    )
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

Train rows: 8935676
Test rows: 905702


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
X_train = df.iloc[train_idx][selected_features].copy()
X_test = df.iloc[test_idx][selected_features].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8935676, 8)
X_test: (905702, 8)


In [10]:
y_train = df.iloc[train_idx]["ctr"].copy()
y_test = df.iloc[test_idx]["ctr"].copy()

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining CTR:")
print(y_train.describe())

print("\nTest CTR:")
print(y_test.describe())

y_train shape: (8935676,)
y_test shape: (905702,)

Training CTR:
count    8.935676e+06
mean     1.156975e-03
std      1.873814e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
Name: ctr, dtype: float64

Test CTR:
count    905702.000000
mean          0.000868
std           0.013038
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: ctr, dtype: float64


In [11]:
print("Missing y_train:", y_train.isna().sum())
print("Missing y_test:", y_test.isna().sum())

Missing y_train: 0
Missing y_test: 0


In [14]:
# Train XGBoost model

model.fit(
    X_train,
    y_train
)

print("Model training completed successfully.")

Model training completed successfully.


In [15]:
predictions = model.predict(X_test)

print("Predictions created.")
print("Number of predictions:", len(predictions))
print(predictions[:10])

Predictions created.
Number of predictions: 905702
[-1.3787117e-06 -1.3787117e-06 -1.3787117e-06 -1.3787117e-06
 -1.3787117e-06 -1.3787117e-06 -1.3787117e-06 -1.3787117e-06
 -1.3787117e-06 -1.3787117e-06]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [20]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    ndcg_score
)
import numpy as np

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(
    mean_squared_error(y_test, predictions)
)

r2 = r2_score(y_test, predictions)

model_ndcg = ndcg_score(
    [y_test.values],
    [predictions]
)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)
print("NDCG:", model_ndcg)

MAE : 0.0015521225596255037
RMSE: 0.013821203764206555
R²  : -0.12372475807550631
NDCG: 0.6947153688294162


In [16]:
# Error analysis

error_df = pd.DataFrame({
    "actual_score": y_test,
    "predicted_score": predictions
})

error_df["absolute_error"] = (
    error_df["actual_score"]
    - error_df["predicted_score"]
).abs()

print("Largest prediction errors:")
print(
    error_df
    .sort_values("absolute_error", ascending=False)
    .head(10)
)


Largest prediction errors:
         actual_score  predicted_score  absolute_error
2901642           1.0         0.000088        0.999912
4507808           1.0         0.001364        0.998636
2501365           1.0         0.001364        0.998636
8988618           1.0         0.001364        0.998636
142898            1.0         0.001927        0.998073
3480586           1.0         0.001927        0.998073
3196405           1.0         0.002328        0.997672
8822693           1.0         0.002328        0.997672
1710253           1.0         0.002532        0.997468
6358333           1.0         0.002550        0.997450


In [17]:
# Feature importance

feature_importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print("Feature importance:")
print(feature_importance)

Feature importance:
                    Feature  Importance
0           gsc_impressions    0.239940
2              ga4_sessions    0.220195
1          gsc_avg_position    0.183301
3      ga4_engaged_sessions    0.085233
7           position_bucket    0.084237
5           engagement_rate    0.079886
4  ga4_total_engagement_sec    0.054164
6        avg_engagement_sec    0.053045


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.